# Aula 08 - Relacionamentos (JOIN) e Transações

Bancos de dados **relacionais** têm esse nome por um motivo: nós dividimos dados em tabelas separadas para evitar repetição (redundância) e depois as **relacionamos** (juntamos).

Exemplo: Se você tem 1000 funcionários no 'Setor de TI', você não quer escrever 'Setor de TI' 1000 vezes na tabela de Funcionários. Você cria uma tabela de Departamento (com ID 1), e no funcionário você só diz: 'Ele pertence ao Departamento ID 1'.

O `JOIN` é a mágica de juntar essas informações na hora da pesquisa.

In [ ]:
import sqlite3
# Usamos o ':memory:' para criar um banco volátil rápido na memória (sem criar arquivo)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

## 1. Criando Tabelas Relacionadas (Foreign Key)

In [ ]:
# Tabela de Departamentos
cursor.execute('''
CREATE TABLE departamentos (
    id INTEGER PRIMARY KEY, 
    nome TEXT
)
''')

# Tabela de Funcionários referenciando (apontando) pro Departamento
cursor.execute('''
CREATE TABLE funcionarios (
    id INTEGER PRIMARY KEY, 
    nome TEXT, 
    id_departamento INTEGER, 
    FOREIGN KEY(id_departamento) REFERENCES departamentos(id)
)
''')

## 2. Transações (O Tudo ou Nada)
Se você vai transferir dinheiro do banco, duas coisas precisam acontecer: 1. Tirar seu saldo. 2. Colocar na conta do amigo. Se a luz cair no meio da operação 2, você perdeu dinheiro! 
As **Transações** agrupam comandos. Ou TUDO dá certo (`commit`), ou o banco desfaz TUDO (`rollback`).

In [ ]:
try:
    # Operação 1: Inserindo departamentos
    cursor.execute("INSERT INTO departamentos (nome) VALUES ('TI'), ('RH')")
    
    # Operação 2: Inserindo funcionários (Alice é do TI=1, Carlos é do RH=2)
    cursor.execute("INSERT INTO funcionarios (nome, id_departamento) VALUES ('Alice', 1), ('Carlos', 2)")
    
    # Deu tudo certo? Confirmamos e salvamos! (Commit)
    conn.commit()
    print('Transação salva com sucesso!')
except Exception as erro:
    # Deu algum erro na linha 1 ou 2? Desfazemos tudo e não salvamos nada! (Rollback)
    conn.rollback()
    print('Erro! Tudo foi desfeito. Motivo:', erro)

## 3. O Famoso JOIN (Juntando as Tabelas)

In [ ]:
# Queremos ver o Nome da pessoa e o Nome do departamento.
# A palavra INNER JOIN junta as linhas de Funcionários com Departamentos onde o 'id_departamento' bater com o 'id'.
comando_join = '''
SELECT 
    funcionarios.nome,
    departamentos.nome
FROM 
    funcionarios
INNER JOIN 
    departamentos ON funcionarios.id_departamento = departamentos.id
'''

cursor.execute(comando_join)
resultados = cursor.fetchall()

print('\nLista de Pessoas e seus Departamentos reais:')
for pessoa, depto in resultados:
    print(f'-> {pessoa} trabalha no {depto}')

conn.close()